In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from typing import List, Tuple, Dict
import time
import warnings
warnings.filterwarnings('ignore')

# Import our optimization package
from bayesian_optimizer import BayesianOptimizer
from objective_function import MockObjectiveFunction

# Set up plotting
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# Global variables for storing results
optimization_results = []
current_optimizer = None
current_objective = None

print("Interactive Bayesian Optimization Setup Complete!")
print("Now you can configure and run optimizations in the cells below.")


In [ ]:
# EXPERIMENT CONFIGURATION - MODIFY THESE VALUES!

# Optimization Settings
BATCH_SIZE = 8          # Number of candidates per iteration (try: 4, 8, 12, 16)
N_ITERATIONS = 15       # Number of optimization iterations (try: 10, 15, 20, 30)
N_INITIAL_POINTS = 10   # Initial random samples (try: 5, 10, 15, 20)
NOISE_STD = 0.1         # Objective function noise level (try: 0.05, 0.1, 0.2)
SEED = 42               # Random seed for reproducibility (try: 42, 123, 456)

# Experiment Name (for tracking different runs)
EXPERIMENT_NAME = "default_run"

# Display configuration
print("CURRENT CONFIGURATION")
print("=" * 50)
print(f"Experiment Name: {EXPERIMENT_NAME}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Iterations: {N_ITERATIONS}")
print(f"Initial Points: {N_INITIAL_POINTS}")
print(f"Noise Level: {NOISE_STD}")
print(f"Random Seed: {SEED}")
print(f"Total Evaluations: {N_INITIAL_POINTS + N_ITERATIONS * BATCH_SIZE}")
print("=" * 50)
print("=->Change values above and re-run this cell to update configuration!")


In [ ]:
def run_optimization_experiment():
    """Run a complete optimization experiment with current settings."""
    global current_optimizer, current_objective
    
    print(f"Starting Run: {EXPERIMENT_NAME}")
    print("=" * 60)
    
    # Define parameter bounds
    bounds = torch.tensor([
            [2, 20],      # concentration
            [0.01, 20],     # print_speed
            [50.0, 100.0],   # gap_size
            [6.0, 12.0]      # volume
    ]).T
    
    # Create optimizer and objective function
    current_optimizer = BayesianOptimizer(
        bounds=bounds,
        batch_size=BATCH_SIZE,
        noise_variance=0.01,
        seed=SEED
    )
    
    current_objective = MockObjectiveFunction(noise_std=NOISE_STD, seed=SEED)
    
    # Get true optimum for comparison
    true_params, _ = current_objective.get_optimal_parameters()
    true_score = current_objective.evaluate_at_optimal()
    
    print(f"True optimal score: {true_score:.4f}")
    print(f"Starting optimization...")
    
    # Run optimization
    start_time = time.time()
    
    best_params, best_score = current_optimizer.optimize(
        objective_function=current_objective,
        n_iterations=N_ITERATIONS,
        n_initial_points=N_INITIAL_POINTS,
        verbose=False  # Reduce output for cleaner notebook
    )
    
    end_time = time.time()
    optimization_time = end_time - start_time
    
    # Store results
    result = {
        'experiment_name': EXPERIMENT_NAME,
        'batch_size': BATCH_SIZE,
        'n_iterations': N_ITERATIONS,
        'n_initial_points': N_INITIAL_POINTS,
        'noise_std': NOISE_STD,
        'seed': SEED,
        'best_params': best_params,
        'best_score': best_score,
        'true_score': true_score,
        'optimization_time': optimization_time,
        'total_evaluations': len(current_optimizer.train_X),
        'optimizer': current_optimizer,
        'objective': current_objective
    }
    
    optimization_results.append(result)
    
    print("\nOPTIMIZATION COMPLETE")
    print("=" * 60)
    print(f"Time: {optimization_time:.2f} seconds")
    print(f"Total evaluations: {result['total_evaluations']}")
    print(f"Best score: {best_score:.4f}")
    print(f"True optimal: {true_score:.4f}")
    print(f"Gap: {(true_score - best_score):.4f}")
    
    # Parameter comparison
    param_names = ['concentration', 'print_speed', 'gap_size', 'volume']
    print(f"\nBest Parameters Found:")
    for i, (name, value) in enumerate(zip(param_names, best_params)):
        print(f"  {name}: {value:.3f}")
    
    return result

# Run the experiment
experiment_result = run_optimization_experiment()


In [ ]:
def plot_optimization_results(optimizer, objective):
    """Create comprehensive plots of the optimization results."""
    
    history = optimizer.get_optimization_history()
    train_X, train_Y = optimizer.get_training_data()
    true_params, _ = objective.get_optimal_parameters()
    true_score = objective.evaluate_at_optimal()
    
    # Create the visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. Optimization Progress
    iterations = [h['iteration'] for h in history]
    best_values = [h['best_value'] for h in history]
    
    axes[0, 0].plot(iterations, best_values, 'b-o', linewidth=2, markersize=6)
    axes[0, 0].axhline(y=true_score, color='r', linestyle='--', alpha=0.7, 
                       label=f'True optimum: {true_score:.3f}')
    axes[0, 0].set_xlabel('Iteration')
    axes[0, 0].set_ylabel('Best Observed Value')
    axes[0, 0].set_title('Optimization Progress')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Improvement Rate
    improvements = [best_values[i] - best_values[0] for i in range(len(best_values))]
    axes[0, 1].plot(iterations, improvements, 'g-o', linewidth=2, markersize=6)
    axes[0, 1].set_xlabel('Iteration')
    axes[0, 1].set_ylabel('Improvement from Initial')
    axes[0, 1].set_title('Cumulative Improvement')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Parameter Space Exploration (Concentration vs Print Speed)
    scatter = axes[0, 2].scatter(train_X[:, 0], train_X[:, 1], c=train_Y.squeeze(), 
                                cmap='viridis', alpha=0.6, s=50)
    axes[0, 2].scatter(optimizer.best_parameters[0], optimizer.best_parameters[1], 
                      c='red', s=200, marker='*', label='Best found', 
                      edgecolor='black', linewidth=2)
    axes[0, 2].scatter(true_params[0], true_params[1], c='orange', s=200, marker='*', 
                      label='True optimum', edgecolor='black', linewidth=2)
    axes[0, 2].set_xlabel('Concentration')
    axes[0, 2].set_ylabel('Print Speed (mm/s)')
    axes[0, 2].set_title('Parameter Space Exploration')
    axes[0, 2].legend()
    plt.colorbar(scatter, ax=axes[0, 2], label='Objective Value')
    
    # 4. Gap Size vs Volume
    scatter2 = axes[1, 0].scatter(train_X[:, 2], train_X[:, 3], c=train_Y.squeeze(), 
                                 cmap='viridis', alpha=0.6, s=50)
    axes[1, 0].scatter(optimizer.best_parameters[2], optimizer.best_parameters[3], 
                      c='red', s=200, marker='*', label='Best found', 
                      edgecolor='black', linewidth=2)
    axes[1, 0].scatter(true_params[2], true_params[3], c='orange', s=200, marker='*', 
                      label='True optimum', edgecolor='black', linewidth=2)
    axes[1, 0].set_xlabel('Gap Size (mm)')
    axes[1, 0].set_ylabel('Volume (μL)')
    axes[1, 0].set_title('🔍 Gap Size vs Volume')
    axes[1, 0].legend()
    plt.colorbar(scatter2, ax=axes[1, 0], label='Objective Value')
    
    # 5. Objective Value Distribution
    axes[1, 1].hist(train_Y.squeeze().numpy(), bins=20, alpha=0.7, color='skyblue', 
                   edgecolor='black')
    axes[1, 1].axvline(optimizer.best_observed_value, color='red', linestyle='--', 
                      linewidth=2, label=f'Best: {optimizer.best_observed_value:.3f}')
    axes[1, 1].axvline(true_score, color='orange', linestyle='--', linewidth=2, 
                      label=f'True: {true_score:.3f}')
    axes[1, 1].set_xlabel('Objective Value')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Objective Value Distribution')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # 6. Parameter Convergence
    param_names = ['concentration', 'print_speed', 'gap_size', 'volume']
    eval_order = np.arange(len(train_X))
    
    for i, name in enumerate(param_names):
        color = plt.cm.Set1(i)
        axes[1, 2].scatter(eval_order, train_X[:, i], alpha=0.6, s=30, 
                          c=color, label=name)
        axes[1, 2].axhline(y=true_params[i], color=color, linestyle='--', alpha=0.7)
    
    axes[1, 2].set_xlabel('Evaluation Order')
    axes[1, 2].set_ylabel('Parameter Value')
    axes[1, 2].set_title('Parameter Convergence')
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return fig

# Create the plots
if current_optimizer is not None:
    fig = plot_optimization_results(current_optimizer, current_objective)
    print("Plots created successfully!")
else:
    print("No optimization results to plot. Run the optimization first!")
